# Instagram Post Downloader (standalone)

Downloads a fixed list of Instagram posts (videos, images, carousels) using **your own** Instagram account.

**What you need:**
- This notebook + `posts_to_download.csv` (must sit in the **same folder**).
- Your Instagram **username and password** (entered below, password is hidden).

**How it works:** it logs in with `instaloader`, then downloads each post into `downloaded/`. Media + metadata `.json` are saved per post.

**Important — avoid getting rate-limited:**
- Run the cells **in order, once**.
- If you see *"Please wait a few minutes"* / a 401, **stop**, wait 30–60 min (or switch network / use a phone hotspot), then run again. It will skip posts already downloaded.
- The script paces itself (pauses between posts) on purpose. Don't remove the delays.

## 1. Install dependencies

Installs the two packages this notebook needs:
- **`instaloader`** — logs into Instagram and downloads posts (videos, photos, and carousels) through your account.
- **`pandas`** — reads the `posts_to_download.csv` list.

Run this once. If they're already installed it finishes instantly. A few warning lines from `pip` are normal and safe to ignore.

In [ ]:
import sys, subprocess
for pkg in ["instaloader", "pandas"]:
    print(f"Installing {pkg}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("\nDone.")


## 2. Enter your Instagram credentials

Type your **own** Instagram username and password when the prompts appear below the cell.

- The password uses a **hidden prompt** (`getpass`) — it won't be visible on screen and is **never saved** into the notebook file.
- Use a **normal, active account**. Brand-new or empty accounts get blocked by Instagram much faster.
- If your account has **two-factor authentication (2FA)**, instaloader will ask for the one-time code in step 4.

In [ ]:
import getpass

IG_USERNAME = input("Instagram username: ").strip()
IG_PASSWORD = getpass.getpass("Instagram password (hidden): ")
print(f"Will log in as: {IG_USERNAME}")


## 3. Load the list of posts to download

Reads `posts_to_download.csv` (which ships next to this notebook) into a table called `todo`.

Each row is one post to fetch, with columns:
- **`shortcode`** — the post's ID in its URL (`instagram.com/p/<shortcode>/`).
- **`permalink`** — the full link to the post.
- **`media_type`** — `IMAGE`, `CAROUSEL_ALBUM`, or `VIDEO`; decides which sub-folder it's saved into.

If you get a `FileNotFoundError`, the CSV isn't in the same folder as the notebook — copy it next to this file and re-run.

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("posts_to_download.csv")
if not csv_path.exists():
    raise FileNotFoundError(
        "posts_to_download.csv not found. Put it in the same folder as this notebook."
    )

todo = pd.read_csv(csv_path)
print(f"Posts to download: {len(todo)}")
print(todo["media_type"].value_counts().to_string())
todo.head()


## 4. Log in

Logs into Instagram with the credentials from step 2 and checks the session is healthy before downloading anything.

- On the **first run** it logs in and saves a session file, so later runs reuse it instead of logging in again (fewer logins = less chance of being flagged). If you have **2FA**, enter the code when prompted here.
- The `test_login()` check catches the most common problem up front: if Instagram is **rate-limiting** you, the cell stops with a clear message instead of failing halfway through the downloads.
- **If it says you're rate-limited:** wait 30–60 minutes (or switch to a different network / phone hotspot) and run this cell again.

In [ ]:
import instaloader

L = instaloader.Instaloader(
    download_pictures=True,
    download_videos=True,
    download_video_thumbnails=False,
    download_geotags=False,
    download_comments=False,
    save_metadata=True,
    compress_json=False,
    max_connection_attempts=1,
)

# Reuse a saved session if you have logged in before, otherwise log in fresh
try:
    L.load_session_from_file(IG_USERNAME)
    print(f"Loaded existing session for {IG_USERNAME}")
except FileNotFoundError:
    L.login(IG_USERNAME, IG_PASSWORD)
    L.save_session_to_file()
    print(f"Logged in and saved session for {IG_USERNAME}")

if L.test_login() is None:
    raise SystemExit(
        "Instagram is rate-limiting this session right now. "
        "Wait 30-60 min (or switch network) and run this cell again."
    )
print("Session is authenticated.")


## 5. Download

Goes through every post in `todo` and saves it into `downloaded/<image|carousel|video>/<shortcode>/` along with a `.json` metadata file.

What to expect:
- **It's slow on purpose.** There's a 20–40 second pause between posts to look human and avoid getting blocked — **don't remove the delays**. For ~24 posts, budget roughly 10–15 minutes.
- **Safe to re-run.** Posts that already have media on disk are skipped, so if it stops you can just run this cell again and it picks up where it left off.
- **If it hits a rate-limit (401 / "wait a few minutes")** it stops immediately so it doesn't make the block worse. Wait, then re-run.
- **`still_failed.csv`** lists anything that didn't download. Some posts may be genuinely **deleted or private** and can't be recovered by anyone — those are expected to stay in this list.

In [ ]:
import time, random
from datetime import datetime

OUT_DIR = Path("downloaded")
MEDIA_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".mp4"}

def already_done(folder: Path) -> bool:
    return folder.exists() and any(f.suffix.lower() in MEDIA_EXTS for f in folder.iterdir())

succeeded, skipped, still_failed = [], [], []

for n, row in enumerate(todo.itertuples(index=False), 1):
    shortcode = str(row.shortcode)
    media_type = str(row.media_type)
    sub = {"VIDEO": "video", "CAROUSEL_ALBUM": "carousel"}.get(media_type, "image")
    target = OUT_DIR / sub / shortcode

    print(f"[{n}/{len(todo)}] {shortcode} ({media_type})")
    if already_done(target):
        print("   already downloaded - skipping")
        skipped.append(shortcode)
        continue

    target.mkdir(parents=True, exist_ok=True)
    try:
        post = instaloader.Post.from_shortcode(L.context, shortcode)
        L.download_post(post, target=target)
        if already_done(target):
            print("   downloaded")
            succeeded.append(shortcode)
        else:
            print("   no media produced")
            still_failed.append({"shortcode": shortcode, "reason": "no media produced"})
    except Exception as e:
        msg = str(e)
        print(f"   failed: {msg}")
        still_failed.append({"shortcode": shortcode, "reason": msg})
        if "401" in msg or "few minutes" in msg.lower():
            print("   >> Rate-limited. Stopping. Wait 30-60 min and re-run this cell.")
            break
    time.sleep(random.uniform(20, 40))  # gentle pacing - do not remove

print(f"\nDone. downloaded={len(succeeded)}  skipped={len(skipped)}  failed={len(still_failed)}")
if still_failed:
    pd.DataFrame(still_failed).to_csv("still_failed.csv", index=False)
    print("Failed ones written to still_failed.csv")


## 6. (Optional) Zip the results to send back

Bundles everything in `downloaded/` into a single `downloaded_posts.zip` that's easy to share.

Run this **after** step 5 finishes (or after the last re-run, once `still_failed.csv` is empty or only deleted/private posts remain). Then send `downloaded_posts.zip` back to the person who gave you this notebook.

In [ ]:
import shutil
if Path("downloaded").exists():
    archive = shutil.make_archive("downloaded_posts", "zip", "downloaded")
    print(f"Created {archive}")
    print("Send this zip back to your friend.")
else:
    print("Nothing downloaded yet.")
